# import libraries

In [1]:
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties
import matplotlib.pyplot as plt
from ax.modelbridge.factory import Models
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy   
import time


import sys
sys.path.append('../')
import helper_functions as hf

# generate recommendations

In [2]:
n = hf.get_iteration_number()

for i in range(3):
    print("Very Important: Please Confirm the Iteration Number is Iteration " + str(n))

Very Important: Please Confirm the Iteration Number is Iteration 2
Very Important: Please Confirm the Iteration Number is Iteration 2
Very Important: Please Confirm the Iteration Number is Iteration 2


In [4]:
time_start = time.time()


df_design, ax_client = hf.run_optimizer(current_iteration=n, drug = "IBP", bopt = 1, n_trials=6)

time_end = time.time()
time_duration = round((time_end - time_start)/60,2)

print("Time taken for optimization: " + str(time_duration) + " mins")
print("Time taken for optimization: " + str(time_duration * 60) + " seconds")

[INFO 06-19 14:01:34] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.


**************************************************************************************************************

Generating Bayesian Optimization trialsfor
Drug name:  Ibuprofen IBP  | Iteration:  2

**************************************************************************************************************


[INFO 06-19 14:02:42] ax.service.ax_client: Generated new trial 12 with parameters {'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 0, 's2': 0, 's3': 0, 's4': 100, 's5': 0, 's6': 0, 's7': 100, 's8': 100, 'surfactant_conc': 1, 'drug_conc': 100} using model SAASBO.
/opt/anaconda3/envs/drug_surfactant/lib/python3.11/site-packages/ax/core/data.py:295: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))
[INFO 06-19 14:03:16] ax.service.ax_client: Generated new trial 13 with parameters {'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 0, 's2': 0, 's3': 100, 's4': 100, 's5': 0, 's6': 0, 's7': 0, 's8': 0, 'surfactant_conc': 1, 'drug_conc': 100} using model SAASBO.
/opt/anacon

Time taken for optimization: 4.42 mins
Time taken for optimization: 265.2 seconds


# process results

In [5]:
ax_client = hf.load_design_optimizer(n)
ax_client.experiment.trials

[INFO 06-19 14:06:12] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.


{0: Trial(experiment_name='drug_surfactant', index=0, status=TrialStatus.COMPLETED, arm=Arm(name='0_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 47, 's2': 59, 's3': 49, 's4': 31, 's5': 96, 's6': 8, 's7': 9, 's8': 35, 'surfactant_conc': 85, 'drug_conc': 77})),
 1: Trial(experiment_name='drug_surfactant', index=1, status=TrialStatus.COMPLETED, arm=Arm(name='1_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 95, 's2': 22, 's3': 76, 's4': 63, 's5': 42, 's6': 71, 's7': 55, 's8': 99, 'surfactant_conc': 35, 'drug_conc': 3})),
 2: Trial(experiment_name='drug_surfactant', index=2, status=TrialStatus.COMPLETED, arm=Arm(name='2_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 52, 's2': 98, 's3': 4, 's4': 13, 's5': 22, 's6': 89, 's7': 28, 's8': 56, 'surfactant_conc': 18, 'drug_conc': 33})),
 3: Trial(experiment_name='drug_surfactant', index=3, status=TrialStatus.COMPLETED, arm=Arm(name='3_0', parameter

In [6]:
df_conc, df_vol = hf.design_to_conc_to_vol (n)

In [8]:
plate_well = input("Enter the plate well starting well (e.g., F1): ")
deepplate_well = input("Enter the deep plate well starting well (e.g., F1): ")


print("Please confirm the following information:")
print("Wellplate will start at: " + plate_well)
print("Deep plate will start at: " + deepplate_well)

print()
print("*******************************************************")
print("Continue if correct, or rerun this cell if incorrect.")
print("*******************************************************")

Please confirm the following information:
Wellplate will start at: A1
Deep plate will start at: C1

*******************************************************
Continue if correct, or rerun this cell if incorrect.
*******************************************************


In [9]:

hf.generate_protocol(df_vol=df_vol, iteration=n, plate_well=plate_well, deepplate_well=deepplate_well)

✅ Successfully wrote to: protocol/otflex_2.py


In [10]:
df_absorbance = hf.process_absorbance(iteration=n, threshold=0.1)
df_absorbance

,trial_index,success
0,0,1
1,1,0
2,2,0
3,3,0
4,4,1
5,5,1


In [11]:
results = hf.build_results(n, df_conc, df_absorbance)
results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_conc,drug_conc,success,micelle_drug_conc,complexity
0,12,0,0,0,100,0,0,100,100,0.5,25.00,1,2.500,3
1,13,0,0,100,100,0,0,0,0,0.5,25.00,0,0.000,2
2,14,0,0,0,100,100,100,0,0,0.5,25.00,0,0.000,3
3,15,0,0,0,100,0,100,0,100,0.5,25.00,0,0.000,3
4,16,0,0,100,100,0,0,100,100,0.5,25.00,1,2.500,4
5,17,0,0,0,100,100,100,100,100,0.5,0.25,1,0.025,5


In [12]:
norm_results = hf.normalize_data(results, 'normalize')

In [13]:
norm_results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_conc,drug_conc,success,micelle_drug_conc,complexity
0,12,0,0,0,100,0,0,100,100,0.01,25.00,1,1.00,0.375
1,13,0,0,100,100,0,0,0,0,0.01,25.00,0,0.00,0.250
2,14,0,0,0,100,100,100,0,0,0.01,25.00,0,0.00,0.375
3,15,0,0,0,100,0,100,0,100,0.01,25.00,0,0.00,0.375
4,16,0,0,100,100,0,0,100,100,0.01,25.00,1,1.00,0.500
5,17,0,0,0,100,100,100,100,100,0.01,0.25,1,0.01,0.625


# load the results to the optimizer

In [14]:
ax_client = hf.load_data_to_optimizer(iteration = n, norm_results = norm_results)
ax_client

[INFO 06-19 14:55:56] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
[INFO 06-19 14:55:56] ax.service.ax_client: Completed trial 12 with data: {'micelle_drug_conc': (1.0, None), 'surfactant_conc': (0.01, None), 'complexity': (0.375, None)}.
[INFO 06-19 14:55:56] ax.service.ax_client: Completed trial 13 with data: {'micelle_drug_conc': (0.0, None), 'surfactant_conc': (0.01, None), 'complexity': (0.25, None)}.
[INFO 06-19 14:55:56] ax.service.ax_client: Completed trial 14 with data: {'micelle_drug_conc': (0.0, None), 'surfactant_conc': (0.01, None), 'complexity': (0.375, None)}.
[INFO 06-19 14:55:56] ax.service.ax_client: Completed trial 15 with data: {'micelle_drug_conc': (0.0, None), 'surfactant_conc': (0.01, None), 'complexity': (0.375, None)}.
[INFO 06-19 14:55:56] ax.service.ax_client: Completed trial 16 with data: {'micelle_drug_c

AxClient(experiment=Experiment(drug_surfactant))